# ISIC 2019 — ResNet50, SVM và thử nghiệm loss

Mục tiêu: đánh giá việc thay bộ phân loại bằng SVM và thay loss,
trong khi giữ nguyên tiền xử lý resize và split của baseline.

- B00: ResNet50 + CrossEntropy thường, đã huấn luyện và khóa.
- SVM: dùng đặc trưng ResNet50 B00 đóng băng để học Linear SVM.
- L01: ResNet50 + weighted CrossEntropy, huấn luyện độc lập.

Chọn cấu hình bằng validation macro-F1; chưa dùng test để lựa chọn.

## 1. Import thư viện

In [1]:
from pathlib import Path
import os, json, random, hashlib, shutil, time, zipfile
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import cv2
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import models, transforms
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score, balanced_accuracy_score
from IPython.display import display

cv2.setNumThreads(1)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)


Device: cuda


## 2. Cấu hình cố định — khóa trước thí nghiệm
Không thay loss, optimizer, augmentation, seed hoặc split riêng cho A1. Nếu cần đổi, tạo một phiên bản thí nghiệm mới.

In [2]:
INPUT_ROOT = Path('/kaggle/input')
DATA_ROOT = None  # Nếu có nhiều CSV trùng tên: đặt đường dẫn dataset ISIC cụ thể.
IMAGE_ROOT = None  # Mặc định quét trong dataset chứa GroundTruth.
OUT = Path('/kaggle/working/ISIC2019_R50_CE_701020_v1')
# Tìm Output baseline đã gắn trong Input.
baseline_folders = [
    p.parent
    for p in INPUT_ROOT.rglob("split_manifest.csv")
    if (p.parent / "B00_R50" / "complete.json").is_file()
    and (p.parent / "B00_R50" / "best_resnet50.pt").is_file()
]

assert len(baseline_folders) == 1, (
    f"Cần đúng một Output baseline đầy đủ; "
    f"tìm thấy {len(baseline_folders)}."
)

RESUME_ROOT = str(baseline_folders[0])
print("Dùng lại baseline:", RESUME_ROOT)
A1_REVIEWED = False
A1_REVIEW_NOTE = ''  # Ghi các ID đã xem + nhận xét bờ, màu, chi tiết bị sửa.
RUN_TEST = False
METHOD_FROZEN_NOTE = ''

LABELS = ['MEL', 'NV', 'BCC', 'AK', 'BKL', 'DF', 'VASC', 'SCC']
SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 16
PATIENCE = 4
NUM_WORKERS = 2
LR = 1e-4
WEIGHT_DECAY = 1e-4
HAIR = dict(kernel=9, threshold=20, min_area=8, max_area=500,
            min_length=12, min_elongation=4.0, max_width=5.0,
            max_fraction=0.03, radius=2)
OUT.mkdir(parents=True, exist_ok=True)

def sha_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def dump_json(path, data):
    Path(path).write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding='utf-8')

def reset_seed():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

reset_seed()


Dùng lại baseline: /kaggle/input/notebooks/tuanducnguyen123/isic2019-a1-v3-quality-check-kh-l-ng/ISIC2019_R50_CE_701020_v1


## 3. Chọn baseline và thư mục kết quả

Tái sử dụng checkpoint và split của B00 ResNet50 đã khóa.
Kết quả SVM và loss mới được lưu riêng; chưa sử dụng test.

In [3]:
## Cell S1 — Chọn B00 và Output

# S1 — chạy ngay sau mục 1–2 cũ, trước mục 3–4 cũ.
BASELINE_INPUT = None  # Điền thư mục B00 nếu có nhiều Output cùng loại.
if BASELINE_INPUT is None:
    found=[p.parent for p in INPUT_ROOT.rglob('protocol_lock.json')
           if (p.parent/'B00_R50/complete.json').is_file()
           and (p.parent/'B00_R50/best_resnet50.pt').is_file()]
    assert len(found)==1,f'Cần 1 Output baseline đầy đủ, tìm thấy {found}'
    BASELINE_INPUT=str(found[0])
B00_ROOT=Path(BASELINE_INPUT)
RESUME_ROOT=str(B00_ROOT)
A1_REVIEWED=False
RUN_TEST=False
NEXT_OUT=Path('/kaggle/working/ISIC2019_R50_SVM_Loss_v1')
NEXT_OUT.mkdir(parents=True,exist_ok=True)
# Nếu khôi phục từ Output đã lưu: trỏ đến thư mục ISIC2019_R50_SVM_Loss_v1.
RESTORE_NEXT_ROOT=None
if RESTORE_NEXT_ROOT:
    src=Path(RESTORE_NEXT_ROOT)
    assert (src/'reference.json').is_file()
    for p in src.rglob('*'):
        if p.is_file():
            dst=NEXT_OUT/p.relative_to(src);dst.parent.mkdir(parents=True,exist_ok=True)
            if dst.exists():assert sha_file(dst)==sha_file(p),f'File khác: {dst}'
            else:shutil.copy2(p,dst)
print('B00:',B00_ROOT,'\nOutput mới:',NEXT_OUT)


B00: /kaggle/input/notebooks/tuanducnguyen123/isic2019-a1-v3-quality-check-kh-l-ng/ISIC2019_R50_CE_701020_v1 
Output mới: /kaggle/working/ISIC2019_R50_SVM_Loss_v1


## 4. Đọc nhãn và kiểm tra dữ liệu
Nhãn lấy từ GroundTruth, không lấy từ tên thư mục. SHA-256 chỉ phát hiện ảnh giống byte; không phát hiện mọi ảnh gần trùng. Các nhóm chứa ảnh giống byte nhưng nhãn mâu thuẫn được loại và lưu báo cáo, không tự đổi nhãn.

In [4]:
root = Path(DATA_ROOT) if DATA_ROOT else INPUT_ROOT

def unique_file(name):
    found = list(root.rglob(name))
    if len(found) != 1:
        raise ValueError(f'Cần đúng 1 {name}; tìm thấy {len(found)}. Đặt DATA_ROOT.')
    return found[0]

truth_path = unique_file('ISIC_2019_Training_GroundTruth.csv')
meta_path = unique_file('ISIC_2019_Training_Metadata.csv')
truth = pd.read_csv(truth_path)
meta = pd.read_csv(meta_path)
assert truth.image.is_unique and meta.image.is_unique
assert len(truth) == 25331, 'Cần đủ bộ train ISIC 2019 chính thức.'
y = truth[LABELS].apply(pd.to_numeric, errors='raise').to_numpy()
assert np.isin(y, [0, 1]).all() and (y.sum(1) == 1).all()
if 'UNK' in truth:
    assert pd.to_numeric(truth.UNK).eq(0).all()
frame = truth[['image']].copy()
frame['label'] = y.argmax(1)
frame['label_name'] = [LABELS[i] for i in frame.label]
frame = frame.merge(meta[['image', 'lesion_id']], on='image', how='left', validate='one_to_one', indicator=True)
assert frame['_merge'].eq('both').all()
frame = frame.drop(columns='_merge')
missing = frame.lesion_id.isna() | frame.lesion_id.astype(str).str.strip().eq('')
frame['group'] = 'LESION_' + frame.lesion_id.astype(str)
frame.loc[missing, 'group'] = 'MISSING_' + frame.loc[missing, 'image']

scan_root = Path(IMAGE_ROOT) if IMAGE_ROOT else truth_path.parent
wanted = set(frame.image)
image_map = {}
for p in scan_root.rglob('*'):
    if p.suffix.lower() in {'.jpg', '.jpeg', '.png'} and p.stem in wanted:
        if p.stem in image_map:
            raise ValueError(f'Trùng đường dẫn image ID {p.stem}; đặt IMAGE_ROOT chính xác.')
        image_map[p.stem] = str(p)
frame['path'] = frame.image.map(image_map)
assert frame.path.notna().all(), 'Thiếu ảnh: kiểm tra IMAGE_ROOT.'
with ThreadPoolExecutor(max_workers=4) as pool:
    frame['file_sha256'] = list(pool.map(sha_file, frame.path))
conflict_hashes = frame.groupby('file_sha256').label.nunique()
bad = frame.file_sha256.isin(conflict_hashes[conflict_hashes > 1].index)
frame[bad].to_csv(OUT / 'conflicting_exact_duplicates.csv', index=False)
removed = int(bad.sum())
frame = frame[~bad].copy().reset_index(drop=True)

# Union-find: cùng lesion hoặc cùng bytes phải nằm cùng tập.
parent = {g: g for g in frame.group.unique()}
def find(g):
    while parent[g] != g:
        parent[g] = parent[parent[g]]
        g = parent[g]
    return g
for _, block in frame.groupby('file_sha256'):
    groups = block.group.unique()
    for g in groups[1:]:
        a, b = find(groups[0]), find(g)
        if a != b:
            parent[max(a, b)] = min(a, b)
frame['group'] = frame.group.map(find)
assert frame.groupby('group').label.nunique().max() == 1, 'Một nhóm có nhãn mâu thuẫn; cần rà soát.'
frame = frame.sort_values('image').reset_index(drop=True)
dump_json(OUT / 'data_audit.json', dict(original=len(truth), retained=len(frame),
    excluded_conflicting_images=removed, missing_lesion_id=int(missing.sum()),
    exact_duplicate_excess=int(frame.file_sha256.duplicated().sum()),
    groundtruth_sha=sha_file(truth_path), metadata_sha=sha_file(meta_path)))
print('Giữ:', len(frame), '| loại ảnh mâu thuẫn:', removed, '| thiếu lesion_id:', int(missing.sum()))


Giữ: 25327 | loại ảnh mâu thuẫn: 4 | thiếu lesion_id: 2084


## 5. Split 70/10/20 theo nhóm và lưu cố định
Dùng 10 fold phân tầng theo nhóm: fold 0–1 test, fold 2 validation, fold 3–9 train. Tỉ lệ ảnh xấp xỉ 70/10/20 do không tách một tổn thương sang hai tập. Không tìm seed/split theo điểm mô hình. Khi chạy lại hoặc tiếp tục phiên, tái sử dụng manifest đã lưu.

In [5]:
manifest_path = OUT / 'split_manifest.csv'
resume = Path(RESUME_ROOT) if RESUME_ROOT else None
if not manifest_path.exists() and resume:
    shutil.copy2(resume / 'split_manifest.csv', manifest_path)
if manifest_path.exists():
    old = pd.read_csv(manifest_path, keep_default_na=False).sort_values('image').reset_index(drop=True)
    for col in ['image', 'label_name', 'group', 'file_sha256']:
        assert old[col].equals(frame[col].reset_index(drop=True)), f'Manifest khác dữ liệu: {col}'
    frame['split'] = old['split']
else:
    folds = np.full(len(frame), -1)
    splitter = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=SEED)
    for fold, (_, idx) in enumerate(splitter.split(frame, frame.label, frame.group)):
        folds[idx] = fold
    frame['split'] = np.where(folds < 2, 'test', np.where(folds == 2, 'val', 'train'))
    frame.drop(columns='path').to_csv(manifest_path, index=False)
assert set(frame.split) == {'train', 'val', 'test'}
for column in ['image', 'group', 'file_sha256']:
    assert frame.groupby(column).split.nunique().max() == 1, f'Leakage: {column}'
parts = {s: frame[frame.split == s].reset_index(drop=True) for s in ['train', 'val', 'test']}
for s, data in parts.items():
    assert set(data.label_name) == set(LABELS), f'{s} thiếu lớp.'
    data.drop(columns='path').to_csv(OUT / f'{s}_manifest.csv', index=False)
distribution = pd.crosstab(frame.label_name, frame.split).reindex(LABELS)[['train', 'val', 'test']]
distribution.to_csv(OUT / 'class_distribution.csv')
display(distribution)
print(frame.split.value_counts(), '\nTỉ lệ:', frame.split.value_counts(normalize=True).round(4).to_dict())


split,train,val,test
label_name,,,
MEL,3077,478,965
NV,9019,1247,2607
BCC,2374,348,601
AK,611,84,172
BKL,1847,285,492
DF,187,20,32
VASC,151,22,80
SCC,457,55,116


split
train    17723
test      5065
val       2539
Name: count, dtype: int64 
Tỉ lệ: {'train': 0.6998, 'test': 0.2, 'val': 0.1002}


## 6. Tiền xử lý B00 và A1
B00 ép ảnh về 224×224, có thể méo ảnh chữ nhật. A1 chỉ thêm khử lông sau cùng phép resize. Black-hat tìm vùng tối nhỏ; lọc thành phần dài/mảnh bằng hình học; Telea nội suy vùng mask. Không phải DullRazor nguyên bản, không phải mask y khoa. Bộ lọc có thể bỏ sót lông cong/giao nhau hoặc nhầm cấu trúc tổn thương; giới hạn 3% diện tích không đảm bảo an toàn.

In [6]:
def preprocess(image, method):
    image = image.convert('RGB').resize((IMG_SIZE, IMG_SIZE), Image.Resampling.BILINEAR)
    rgb = np.asarray(image).copy()
    mask = np.zeros(rgb.shape[:2], dtype=np.uint8)
    if method == 'B00_R50':
        return image, mask, False
    assert method == 'A1_R50'
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (HAIR['kernel'], HAIR['kernel']))
    response = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, kernel)
    raw = (response > HAIR['threshold']).astype(np.uint8)
    count, components, stats, _ = cv2.connectedComponentsWithStats(raw, 8)
    for k in range(1, count):
        area = stats[k, cv2.CC_STAT_AREA]
        if not HAIR['min_area'] <= area <= HAIR['max_area']:
            continue
        yy, xx = np.where(components == k)
        points = np.column_stack([xx, yy]).astype(np.float32)
        (_, _), (w, h), _ = cv2.minAreaRect(points)
        length, width = max(w, h) + 1, min(w, h) + 1
        if length >= HAIR['min_length'] and width <= HAIR['max_width'] and length / width >= HAIR['min_elongation']:
            mask[components == k] = 255
    if np.any(mask):
        mask = cv2.dilate(mask, np.ones((3, 3), np.uint8), iterations=1)
    fallback = float((mask > 0).mean()) > HAIR['max_fraction']
    if fallback or not np.any(mask):
        return image, mask, fallback
    repaired = cv2.inpaint(rgb, mask, HAIR['radius'], cv2.INPAINT_TELEA)
    return Image.fromarray(repaired), mask, fallback


## 7. Dataset và DataLoader
Chuẩn hóa ImageNet là một phần cố định của cả hai lượt. Không WeightedRandomSampler, không class weights, không label smoothing.

In [7]:
normalize = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
class SkinDataset(Dataset):
    def __init__(self, data, method):
        self.data = data.reset_index(drop=True)
        self.method = method
    def __len__(self):
        return len(self.data)
    def __getitem__(self, index):
        row = self.data.iloc[index]
        with Image.open(row.path) as im:
            image, _, _ = preprocess(im, self.method)
        return normalize(image), int(row.label)

def seed_worker(worker_id):
    seed = torch.initial_seed() % (2**32)
    random.seed(seed); np.random.seed(seed)

def make_loader(data, method, train=False):
    generator = torch.Generator().manual_seed(SEED)
    return DataLoader(SkinDataset(data, method), batch_size=BATCH_SIZE,
        shuffle=train, num_workers=NUM_WORKERS, pin_memory=True,
        worker_init_fn=seed_worker, generator=generator, drop_last=False)


## 8. ResNet50, Cross-Entropy và hàm đánh giá
Hai mô hình bắt đầu từ cùng trọng số ImageNet và cùng seed cho lớp cuối; không dùng checkpoint B00 để train A1.

In [8]:
def build_model():
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, len(LABELS))
    return model.to(DEVICE)

criterion = nn.CrossEntropyLoss()  # Không weight; label_smoothing mặc định 0.

def metrics(y, pred):
    return dict(accuracy=float(accuracy_score(y, pred)),
        macro_f1=float(f1_score(y, pred, labels=range(8), average='macro', zero_division=0)),
        balanced_accuracy=float(balanced_accuracy_score(y, pred)))

def evaluate(model, loader):
    model.eval()
    loss_sum, total, labels, probabilities = 0.0, 0, [], []
    with torch.no_grad():
        for images, y in loader:
            images, y = images.to(DEVICE), y.to(DEVICE)
            logits = model(images)
            loss_sum += criterion(logits, y).item() * len(y)
            total += len(y)
            labels.extend(y.cpu().tolist())
            probabilities.append(logits.softmax(1).cpu().numpy())
    probabilities = np.concatenate(probabilities)
    pred = probabilities.argmax(1)
    return loss_sum / total, metrics(labels, pred), np.array(labels), pred, probabilities

def export_evaluation(model, data, method, folder, split):
    loss, scores, y, pred, probs = evaluate(model, make_loader(data, method))
    result = data[['image', 'group', 'file_sha256', 'label_name']].rename(columns={'label_name':'true_label'}).copy()
    result['pred_label'] = [LABELS[i] for i in pred]
    for i, label in enumerate(LABELS): result['p_' + label] = probs[:, i]
    result.to_csv(folder / f'{split}_predictions.csv', index=False)
    report = classification_report(y, pred, labels=range(8), target_names=LABELS, output_dict=True, zero_division=0)
    pd.DataFrame(report).T.to_csv(folder / f'{split}_per_class.csv')
    cm = confusion_matrix(y, pred, labels=range(8))
    pd.DataFrame(cm, index=LABELS, columns=LABELS).to_csv(folder / f'{split}_confusion.csv')
    fig, ax = plt.subplots(figsize=(7, 6)); ax.imshow(cm, cmap='Blues')
    ax.set(xticks=range(8), yticks=range(8), xticklabels=LABELS, yticklabels=LABELS,
           xlabel='Dự đoán', ylabel='Nhãn thật', title=f'{method} — {split}')
    for i in range(8):
        for j in range(8): ax.text(j, i, str(cm[i,j]), ha='center', va='center', color='white' if cm[i,j] > cm.max()/2 else 'black', fontsize=8)
    fig.tight_layout(); fig.savefig(folder / f'{split}_confusion.png', dpi=140); plt.close(fig)
    dump_json(folder / f'{split}_metrics.json', dict(loss=loss, **scores))
    return scores


## 9. Hàm train dùng chung — checkpoint theo validation macro-F1

In [9]:
def train_experiment(method):
    assert DEVICE.type == 'cuda', 'Bật GPU; không train ResNet50 trên CPU.'
    folder = OUT / method
    if not folder.exists() and resume and (resume / method / 'complete.json').exists():
        shutil.copytree(resume / method, folder)
    folder.mkdir(exist_ok=True)
    complete_path = folder / 'complete.json'
    if complete_path.exists():
        saved = json.loads(complete_path.read_text())
        assert saved['lock_sha'] == LOCK_SHA
        assert sha_file(folder / 'best_resnet50.pt') == saved['checkpoint_sha']
        print(method, 'đã hoàn thành; tái sử dụng kết quả, không train lại.')
        return saved
    if (folder / 'history.csv').exists():
        raise RuntimeError('Lượt train trước chưa hoàn thành. Giữ output đó; dùng OUT mới để chạy lại cả protocol, không âm thầm resume optimizer.')
    reset_seed()
    model = build_model()
    # Hash khởi tạo: xác minh hai lượt khởi tạo giống nhau.
    initial = hashlib.sha256()
    for key, value in model.state_dict().items():
        initial.update(key.encode()); initial.update(value.detach().cpu().numpy().tobytes())
    init_sha = initial.hexdigest()
    train_loader = make_loader(parts['train'], method, train=True)
    val_loader = make_loader(parts['val'], method)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    best, wait, history = -1.0, 0, []
    started = time.perf_counter()
    for epoch in range(1, EPOCHS + 1):
        model.train(); loss_sum = 0.0; total = 0
        for images, y in train_loader:
            images, y = images.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = criterion(logits, y)
            loss.backward(); optimizer.step()
            loss_sum += loss.item() * len(y); total += len(y)
        val_loss, scores, _, _, _ = evaluate(model, val_loader)
        history.append(dict(epoch=epoch, train_loss=loss_sum/total, val_loss=val_loss,
                            lr=optimizer.param_groups[0]['lr'], **scores))
        pd.DataFrame(history).to_csv(folder / 'history.csv', index=False)
        print(f'{method} epoch {epoch:02d}: train={loss_sum/total:.4f}, val={val_loss:.4f}, F1={scores["macro_f1"]:.4f}')
        if scores['macro_f1'] > best + 1e-5:
            best, best_epoch, wait = scores['macro_f1'], epoch, 0
            torch.save(model.state_dict(), folder / 'best_resnet50.pt')
        else:
            wait += 1
        scheduler.step()
        if wait >= PATIENCE: break
    model.load_state_dict(torch.load(folder / 'best_resnet50.pt', map_location=DEVICE, weights_only=True))
    scores = export_evaluation(model, parts['val'], method, folder, 'val')
    h = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot(h.epoch, h.train_loss, label='train'); axes[0].plot(h.epoch, h.val_loss, label='val'); axes[0].legend()
    axes[1].plot(h.epoch, h.macro_f1); axes[1].set_title('Validation macro-F1')
    for ax in axes: ax.set_xlabel('Epoch'); ax.grid(alpha=.2)
    fig.tight_layout(); fig.savefig(folder / 'learning_curves.png', dpi=120); plt.close(fig)
    saved = dict(method=method, lock_sha=LOCK_SHA, initial_sha=init_sha,
        checkpoint_sha=sha_file(folder / 'best_resnet50.pt'), best_epoch=best_epoch,
        completed_epochs=len(history), seconds=time.perf_counter()-started, **scores)
    dump_json(complete_path, saved)
    del model, optimizer, train_loader, val_loader
    torch.cuda.empty_cache()
    return saved


## 10. Kiểm tra điều kiện so sánh

Đối chiếu mã ảnh, nhãn, nhóm tổn thương, cấu hình và checkpoint
với B00. Chuẩn bị hàm đánh giá trên cùng tập validation.

In [10]:
# S2 — sau khi chạy mục 3,4,5,8,9,10 cũ. Không chạy mục train B00/A1 cũ.
import inspect, warnings, joblib
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.exceptions import ConvergenceWarning
from threadpoolctl import threadpool_limits

b00_config=json.loads((B00_ROOT/'protocol_lock.json').read_text())
b00_result=json.loads((B00_ROOT/'B00_R50/complete.json').read_text())
assert sha_file(B00_ROOT/'protocol_lock.json')==b00_result['lock_sha']
assert sha_file(B00_ROOT/'B00_R50/best_resnet50.pt')==b00_result['checkpoint_sha']
assert sha_file(B00_ROOT/'split_manifest.csv')==b00_config['split_sha']
assert RUN_TEST is False and A1_REVIEWED is False
cols=['image','label','label_name','group','file_sha256','split']
ref=pd.read_csv(B00_ROOT/'split_manifest.csv',keep_default_na=False)
for name,actual,expected in [('all',frame,ref)]+[(s,parts[s],ref[ref.split==s]) for s in ['train','val','test']]:
    pd.testing.assert_frame_equal(actual[cols].sort_values('image').reset_index(drop=True).astype(str),
        expected[cols].sort_values('image').reset_index(drop=True).astype(str))
for c in ['image','group','file_sha256']:
    assert frame.groupby(c).split.nunique().max()==1,f'Leakage {c}'
for k,v in dict(labels=LABELS,seed=SEED,size=IMG_SIZE,batch=BATCH_SIZE,epochs=EPOCHS,
        patience=PATIENCE,workers=NUM_WORKERS,lr=LR,weight_decay=WEIGHT_DECAY).items():
    assert v==b00_config[k],f'Khác B00: {k}'
assert isinstance(criterion,nn.CrossEntropyLoss) and criterion.weight is None
assert criterion.label_smoothing==0 and criterion.reduction=='mean'
assert len(normalize.transforms)==2
assert isinstance(normalize.transforms[0],transforms.ToTensor)
assert isinstance(normalize.transforms[1],transforms.Normalize)
assert list(normalize.transforms[1].mean)==[0.485,0.456,0.406]
assert list(normalize.transforms[1].std)==[0.229,0.224,0.225]
shared_source='\n'.join(inspect.getsource(f) for f in [reset_seed,SkinDataset.__init__,
    SkinDataset.__getitem__,SkinDataset.__len__,seed_worker,make_loader,build_model,
    metrics,evaluate,export_evaluation,train_experiment])
source_hash=hashlib.sha256(shared_source.encode()).hexdigest()
assert source_hash==(B00_ROOT/'training_source.sha256').read_text().strip(),'Code train khác B00'
assert hashlib.sha256(inspect.getsource(preprocess).encode()).hexdigest()==b00_config['preprocessing_source'],'preprocess cũ đã bị thay'
env=dict(torch=str(torch.__version__),torchvision=str(torchvision.__version__),numpy=np.__version__,
    pandas=pd.__version__,sklearn=__import__('sklearn').__version__,cv2=cv2.__version__,
    pillow=__import__('PIL').__version__,gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
# Cho phép dùng CPU đọc cache và fit SVM; không âm thầm đổi phiên bản thư viện.
for k in env:
    if k!='gpu':assert env[k]==b00_config['environment'][k],f'Môi trường khác B00: {k}={env[k]}'

def next_lock(path,obj):
    path=Path(path)
    if path.exists():assert json.loads(path.read_text())==obj,f'Lock khác: {path}'
    else:dump_json(path,obj)

reference=dict(baseline_lock_sha=b00_result['lock_sha'],checkpoint_sha=b00_result['checkpoint_sha'],
    split_sha=b00_config['split_sha'],training_source_sha=source_hash,labels=LABELS,
    classifier_test_used=False)
next_lock(NEXT_OUT/'reference.json',reference)
base_predictions=pd.read_csv(B00_ROOT/'B00_R50/val_predictions.csv')
assert base_predictions.image.is_unique
truth=parts['val'][['image','group','file_sha256','label_name']].rename(columns={'label_name':'true_label'})
assert set(base_predictions.image)==set(truth.image)
checked=base_predictions.merge(truth,on='image',suffixes=('','_manifest'),validate='one_to_one')
for c in ['group','file_sha256','true_label']:assert checked[c].eq(checked[c+'_manifest']).all()
label_to_id={l:i for i,l in enumerate(LABELS)}
base_scores=metrics(base_predictions.true_label.map(label_to_id),base_predictions.pred_label.map(label_to_id))
for k,v in base_scores.items():assert np.isclose(v,b00_result[k],atol=1e-10,rtol=0)
shutil.copy2(B00_ROOT/'protocol_lock.json',NEXT_OUT/'B00_protocol_lock.json')
shutil.copy2(B00_ROOT/'B00_R50/complete.json',NEXT_OUT/'B00_complete.json')
shutil.copy2(B00_ROOT/'split_manifest.csv',NEXT_OUT/'split_manifest.csv')
print('B00 validation:',base_scores)

# Dùng cho mọi ứng viên, ghép với B00 theo mã ảnh và nhãn.
def next_report(predictions,folder):
    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)
    assert predictions.image.is_unique and set(predictions.image)==set(truth.image)
    check=predictions.merge(truth,on='image',suffixes=('','_manifest'),validate='one_to_one')
    for c in ['group','file_sha256','true_label']:assert check[c].eq(check[c+'_manifest']).all()
    y=predictions.true_label.map(label_to_id);p=predictions.pred_label.map(label_to_id)
    assert y.notna().all() and p.notna().all()
    scores=metrics(y,p)
    predictions.to_csv(folder/'val_predictions.csv',index=False)
    dump_json(folder/'val_metrics.json',scores)
    report=classification_report(y,p,labels=range(8),target_names=LABELS,output_dict=True,zero_division=0)
    pd.DataFrame(report).T.to_csv(folder/'val_per_class.csv')
    cm=confusion_matrix(y,p,labels=range(8));pd.DataFrame(cm,index=LABELS,columns=LABELS).to_csv(folder/'val_confusion.csv')
    fig,ax=plt.subplots(figsize=(7,6));ax.imshow(cm,cmap='Blues')
    ax.set(xticks=range(8),yticks=range(8),xticklabels=LABELS,yticklabels=LABELS,xlabel='Predicted',ylabel='True')
    for i in range(8):
        for j in range(8):ax.text(j,i,str(cm[i,j]),ha='center',va='center',color='white' if cm[i,j]>cm.max()/2 else 'black')
    fig.tight_layout();fig.savefig(folder/'val_confusion.png',dpi=120);plt.close(fig)
    paired=base_predictions.merge(predictions,on='image',suffixes=('_B00','_candidate'),validate='one_to_one')
    b=paired.true_label_B00.eq(paired.pred_label_B00);a=paired.true_label_candidate.eq(paired.pred_label_candidate)
    paired['transition']=np.select([b&a,b&~a,~b&a],['right_to_right','right_to_wrong','wrong_to_right'],default='wrong_to_wrong')
    paired.to_csv(folder/'paired_validation.csv',index=False)
    pd.crosstab(paired.true_label_B00,paired.transition).reindex(LABELS,fill_value=0).to_csv(folder/'transitions_by_class.csv')
    return scores

B00 validation: {'accuracy': 0.7376920047262702, 'macro_f1': 0.5521146626587701, 'balanced_accuracy': 0.5403517144351776}


## 11. Trích xuất đặc trưng bằng ResNet50

Đóng băng ResNet50 B00 và lấy vector 2.048 đặc trưng cho mỗi ảnh
train/validation. Không cập nhật trọng số CNN, không đọc ảnh test.

In [11]:
# S3 — chỉ trích xuất TRAIN + VALIDATION; không mở ảnh test, không cập nhật CNN.
FEATURE_DIR=NEXT_OUT/'features';FEATURE_DIR.mkdir(exist_ok=True)
def extract_features():
    reset_seed()
    model=models.resnet50(weights=None)
    model.fc=nn.Linear(model.fc.in_features,len(LABELS))
    model.load_state_dict(torch.load(B00_ROOT/'B00_R50/best_resnet50.pt',map_location='cpu',weights_only=True),strict=True)
    model=model.to(DEVICE).eval()
    for p in model.parameters():p.requires_grad_(False)
    head=model.fc;model.fc=nn.Identity()
    cache={}
    for split in ['val','train']:
        features=[];labels=[];logits=[]
        # train=False is intentional: no shuffle; matches manifest row order.
        with torch.inference_mode():
            for x,y in make_loader(parts[split],'B00_R50',train=False):
                z=model(x.to(DEVICE));assert z.ndim==2 and z.shape[1]==2048
                features.append(z.cpu().numpy());labels.extend(y.numpy().tolist())
                if split=='val':logits.append(head(z).cpu().numpy())
        X=np.concatenate(features).astype(np.float32);y=np.asarray(labels,dtype=np.int64)
        assert np.array_equal(y,parts[split].label.to_numpy()) and np.isfinite(X).all()
        if split=='val':
            pred=np.concatenate(logits).argmax(1)
            replay=parts['val'][['image']].copy();replay['replayed_label']=[LABELS[i] for i in pred]
            replay=replay.merge(base_predictions[['image','pred_label']],on='image',validate='one_to_one')
            replay['matches']=replay.replayed_label.eq(replay.pred_label)
            replay.to_csv(NEXT_OUT/'B00_replay_validation.csv',index=False)
            assert replay.matches.all(),'Dự đoán B00 không tái lập đúng; gửi CSV mismatch trước khi fit SVM.'
        np.save(FEATURE_DIR/f'{split}_X.npy',X);np.save(FEATURE_DIR/f'{split}_y.npy',y)
        parts[split][cols].to_csv(FEATURE_DIR/f'{split}_manifest.csv',index=False)
        cache[split]=dict(shape=list(X.shape),X_sha=sha_file(FEATURE_DIR/f'{split}_X.npy'),
            y_sha=sha_file(FEATURE_DIR/f'{split}_y.npy'),manifest_sha=sha_file(FEATURE_DIR/f'{split}_manifest.csv'))
        print(split,X.shape)
    del model,head;torch.cuda.empty_cache()
    return cache

feature_protocol=dict(reference=reference,encoder='B00 best ResNet50, frozen incl BN eval',layer='avgpool flattened 2048',
    input='resize224 bilinear + ImageNet normalization',augmentation='none',
    feature_source_sha=hashlib.sha256(inspect.getsource(extract_features).encode()).hexdigest(),
    software={k:v for k,v in env.items() if k!='gpu'})
next_lock(FEATURE_DIR/'protocol.json',feature_protocol)
cache_file=FEATURE_DIR/'complete.json'
if not cache_file.exists():
    assert DEVICE.type=='cuda','Bật GPU T4 cho lượt trích xuất đầu; SVM sau đó dùng CPU.'
    assert env['gpu']==b00_config['environment']['gpu'],'Dùng cùng GPU B00 để kiểm tra tái lập'
    cache=extract_features();dump_json(cache_file,dict(protocol_sha=sha_file(FEATURE_DIR/'protocol.json'),files=cache,device=env['gpu']))
cache=json.loads(cache_file.read_text())
assert cache['protocol_sha']==sha_file(FEATURE_DIR/'protocol.json')
for split in ['train','val']:
    for suffix,key in [('X.npy','X_sha'),('y.npy','y_sha'),('manifest.csv','manifest_sha')]:
        assert sha_file(FEATURE_DIR/f'{split}_{suffix}')==cache['files'][split][key]
    m=pd.read_csv(FEATURE_DIR/f'{split}_manifest.csv',keep_default_na=False)
    pd.testing.assert_frame_equal(m.astype(str),parts[split][cols].reset_index(drop=True).astype(str))
X_train=np.load(FEATURE_DIR/'train_X.npy',mmap_mode='r');y_train=np.load(FEATURE_DIR/'train_y.npy')
X_val=np.load(FEATURE_DIR/'val_X.npy',mmap_mode='r');y_val=np.load(FEATURE_DIR/'val_y.npy')
print('Đặc trưng sẵn sàng. Chỉ encoder được tái dùng; SVM chưa fit.')

val (2539, 2048)
train (17723, 2048)
Đặc trưng sẵn sàng. Chỉ encoder được tái dùng; SVM chưa fit.


## 12. Huấn luyện và đánh giá Linear SVM

Chuẩn hóa đặc trưng bằng thống kê từ train, thử C = 0.1, 1, 10.
Chọn cấu hình theo validation macro-F1 và so sánh với B00.
Điểm SVM không phải xác suất dự đoán.

In [12]:
# S4 — Linear SVM, ba C đã định trước. Fit scaler và SVM CHỈ trên train.
SVM_DIR=NEXT_OUT/'SVM';SVM_DIR.mkdir(exist_ok=True)
SVM_CONFIG=dict(C_values=[0.1,1.0,10.0],class_weight=None,loss='squared_hinge',
    penalty='l2',dual=False,multi_class='ovr',max_iter=5000,tol=1e-4,
    feature_scaling='StandardScaler fit train',selection='val macro-F1; tie -> smaller C',
    seed=SEED,features_complete_sha=sha_file(FEATURE_DIR/'complete.json'),
    sklearn=__import__('sklearn').__version__)
next_lock(SVM_DIR/'config.json',SVM_CONFIG)

def fit_svm_grid(Xtr,ytr,Xv,yv):
    results=[];models_by_C={}
    for C in SVM_CONFIG['C_values']:
        pipe=Pipeline([('scale',StandardScaler()),('svm',LinearSVC(C=C,class_weight=None,
            loss='squared_hinge',penalty='l2',dual=False,multi_class='ovr',
            max_iter=5000,tol=1e-4,random_state=SEED))])
        start=time.perf_counter()
        with warnings.catch_warnings(record=True) as captured,threadpool_limits(limits=2):
            warnings.simplefilter('always',ConvergenceWarning)
            pipe.fit(Xtr,ytr)
        converged=not any(issubclass(w.category,ConvergenceWarning) for w in captured)
        pred=pipe.predict(Xv)
        results.append(dict(C=C,converged=converged,seconds=time.perf_counter()-start,
            iterations=int(np.max(pipe['svm'].n_iter_)),warnings=' | '.join(str(w.message) for w in captured),**metrics(yv,pred)))
        models_by_C[C]=pipe
        print(results[-1])
    return pd.DataFrame(results),models_by_C

next_lock(SVM_DIR/'source_lock.json',dict(source_sha=hashlib.sha256(inspect.getsource(fit_svm_grid).encode()).hexdigest()))
if not (SVM_DIR/'selected.json').exists():
    grid,svm_models=fit_svm_grid(X_train,y_train,X_val,y_val)
    grid.to_csv(SVM_DIR/'svm_search.csv',index=False)
    eligible=grid[grid.converged].sort_values(['macro_f1','C'],ascending=[False,True])
    assert len(eligible),'Chưa có SVM hội tụ. Giữ log và gửi lại; chưa kết luận hiệu năng.'
    best_C=float(eligible.iloc[0].C);chosen=svm_models[best_C]
    joblib.dump(chosen,SVM_DIR/'selected_svm.joblib')
    dump_json(SVM_DIR/'selected.json',dict(C=best_C,model_sha=sha_file(SVM_DIR/'selected_svm.joblib'),
        config_sha=sha_file(SVM_DIR/'config.json'),source_lock_sha=sha_file(SVM_DIR/'source_lock.json')))
    del svm_models
selected=json.loads((SVM_DIR/'selected.json').read_text())
assert selected['config_sha']==sha_file(SVM_DIR/'config.json')
assert selected['source_lock_sha']==sha_file(SVM_DIR/'source_lock.json')
assert selected['model_sha']==sha_file(SVM_DIR/'selected_svm.joblib')
chosen=joblib.load(SVM_DIR/'selected_svm.joblib')  # Chỉ nạp model do chính notebook này tạo.
pred=chosen.predict(X_val);decision=chosen.decision_function(X_val)
assert np.array_equal(chosen['svm'].classes_,np.arange(8))
out=parts['val'][['image','group','file_sha256','label_name']].rename(columns={'label_name':'true_label'}).copy()
out['pred_label']=[LABELS[int(i)] for i in pred]
for i,l in enumerate(LABELS):out['score_'+l]=decision[:,i]  # Điểm SVM, KHÔNG phải xác suất.
svm_scores=next_report(out,SVM_DIR)
display(pd.read_csv(SVM_DIR/'svm_search.csv'))
print('Selected C:',selected['C'],'| SVM:',svm_scores,'| B00:',base_scores)
print('Chênh lệch macro-F1:',100*(svm_scores['macro_f1']-base_scores['macro_f1']),'điểm phần trăm')

{'C': 0.1, 'converged': True, 'seconds': 56.435081210000135, 'iterations': 30, 'warnings': '', 'accuracy': 0.734147302087436, 'macro_f1': 0.5424794521549253, 'balanced_accuracy': 0.5077479947322447}
{'C': 1.0, 'converged': True, 'seconds': 250.14416359999996, 'iterations': 622, 'warnings': '', 'accuracy': 0.7321780228436392, 'macro_f1': 0.5324105072128773, 'balanced_accuracy': 0.49729968847436656}
{'C': 10.0, 'converged': True, 'seconds': 384.4970165949999, 'iterations': 2266, 'warnings': '', 'accuracy': 0.7345411579361953, 'macro_f1': 0.5336399831238439, 'balanced_accuracy': 0.49893695123833337}


,C,converged,seconds,iterations,warnings,accuracy,macro_f1,balanced_accuracy
0,0.1,True,56.435081,30,NaN,0.734147,0.542479,0.507748
1,1.0,True,250.144164,622,NaN,0.732178,0.532411,0.497300
2,10.0,True,384.497017,2266,NaN,0.734541,0.533640,0.498937


Selected C: 0.1 | SVM: {'accuracy': 0.734147302087436, 'macro_f1': 0.5424794521549253, 'balanced_accuracy': 0.5077479947322447} | B00: {'accuracy': 0.7376920047262702, 'macro_f1': 0.5521146626587701, 'balanced_accuracy': 0.5403517144351776}
Chênh lệch macro-F1: -0.9635210503844749 điểm phần trăm


## 13. Thử weighted CrossEntropy — nhánh độc lập

Giữ resize và ResNet50; chỉ thay loss bằng CE có trọng số lớp.
Khởi tạo từ ImageNet, tối đa 16 epoch và patience 4.
Mặc định chưa chạy; bật RUN_L01 khi thực hiện thí nghiệm này.

In [13]:
# L1 — NHÁNH LOSS RIÊNG. Không cần chạy S3/S4; chỉ cần S1,S2 và các mục cũ đã nêu.
# Bật True khi muốn dùng một lượt GPU tối đa 16 epoch, patience 4.
RUN_L01 = True

def preprocess_resize_only(image,method):
    assert method in ['B00_R50','L01_WCE_sqrt_R50']
    image=image.convert('RGB').resize((IMG_SIZE,IMG_SIZE),Image.Resampling.BILINEAR)
    return image,np.zeros((IMG_SIZE,IMG_SIZE),np.uint8),False

def run_weighted_ce():
    assert DEVICE.type=='cuda' and env['gpu']==b00_config['environment']['gpu'],'Dùng GPU như B00'
    counts=parts['train'].label.value_counts().reindex(range(len(LABELS))).to_numpy(dtype=float)
    assert np.all(counts>0)
    weights=1/np.sqrt(counts);weights=weights/weights.mean()
    LROOT=NEXT_OUT/'LOSS_WCE_sqrt';LROOT.mkdir(exist_ok=True)
    weight_table=pd.DataFrame(dict(label=LABELS,train_count=counts.astype(int),weight=weights))
    display(weight_table);weight_table.to_csv(LROOT/'class_weights.csv',index=False)
    config=dict(method='L01_WCE_sqrt_R50',reference=reference,baseline_config=b00_config,
        candidate_loss='CrossEntropy(weight=normalized inverse sqrt train counts)',class_weights=weights.tolist(),
        loss_reporting='sample-count-weighted average of minibatch weighted-CE means; not comparable numerically to unweighted CE',
        train_from='ImageNet IMAGENET1K_V1, NOT B00 checkpoint',max_epochs=EPOCHS,patience=PATIENCE,
        environment=env,source_sha=hashlib.sha256((inspect.getsource(run_weighted_ce)+inspect.getsource(preprocess_resize_only)).encode()).hexdigest())
    next_lock(LROOT/'candidate_protocol_lock.json',config)
    reset_seed();probe=build_model();h=hashlib.sha256()
    for key,value in probe.state_dict().items():
        h.update(key.encode());h.update(value.detach().cpu().numpy().tobytes())
    assert h.hexdigest()==b00_result['initial_sha'],'Khởi tạo khác B00'
    del probe;torch.cuda.empty_cache()
    candidate_criterion=nn.CrossEntropyLoss(weight=torch.tensor(weights,dtype=torch.float32,device=DEVICE))
    missing=object();keys=['OUT','LOCK_SHA','resume','preprocess','criterion']
    old={k:globals().get(k,missing) for k in keys}
    try:
        globals().update(OUT=LROOT,LOCK_SHA=sha_file(LROOT/'candidate_protocol_lock.json'),resume=None,
                         preprocess=preprocess_resize_only,criterion=candidate_criterion)
        result=train_experiment('L01_WCE_sqrt_R50')
    finally:
        for k,v in old.items():
            if v is missing:globals().pop(k,None)
            else:globals()[k]=v
    assert result['initial_sha']==b00_result['initial_sha']
    next_report(pd.read_csv(LROOT/'L01_WCE_sqrt_R50/val_predictions.csv'),LROOT/'comparison')
    return result

if RUN_L01:
    loss_result=run_weighted_ce()
    print('Weighted CE:',loss_result)
else:
    print('L01 chưa chạy. Bật RUN_L01=True để train độc lập; không bắt buộc SVM phải thắng.')

,label,train_count,weight
0,MEL,3077,0.459169
1,NV,9019,0.268199
2,BCC,2374,0.522753
3,AK,611,1.030424
4,BKL,1847,0.592657
5,DF,187,1.862584
6,VASC,151,2.072756
7,SCC,457,1.191458


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 202MB/s]


L01_WCE_sqrt_R50 epoch 01: train=1.1228, val=1.0318, F1=0.5005
L01_WCE_sqrt_R50 epoch 02: train=0.7397, val=0.9815, F1=0.5272
L01_WCE_sqrt_R50 epoch 03: train=0.4821, val=1.1739, F1=0.5113
L01_WCE_sqrt_R50 epoch 04: train=0.3123, val=1.2941, F1=0.4873
L01_WCE_sqrt_R50 epoch 05: train=0.1910, val=1.3508, F1=0.5060
L01_WCE_sqrt_R50 epoch 06: train=0.0925, val=1.6827, F1=0.4886
Weighted CE: {'method': 'L01_WCE_sqrt_R50', 'lock_sha': '57a9ca4d0f87436e082074f147c212c7c184ea4d5c04845855bd7671c685d55f', 'initial_sha': '37a03a16d86fda7f927000366760327e605b16fc8b4cb6a565a4ce763fe9003c', 'checkpoint_sha': '3a4c66eba6955f89f819f925c94dfedaf33545acdd75311145584d43c7cbf921', 'best_epoch': 2, 'completed_epochs': 6, 'seconds': 1394.011746093, 'accuracy': 0.7164237888932651, 'macro_f1': 0.5271856040372417, 'balanced_accuracy': 0.5417949845220715}


## 14. Tổng hợp và xuất báo cáo

Tổng hợp các thí nghiệm đã hoàn thành, chỉ số từng lớp và các
trường hợp chuyển đúng/sai so với B00. Xuất ZIP để phân tích.

In [14]:
# S5 — chạy sau S4 và/hoặc L1. Không chạy test.
comparison=[dict(method='B00_R50_CE',**base_scores)]
for name,path in [('B00_features_LinearSVM',NEXT_OUT/'SVM/val_metrics.json'),
                  ('L01_WCE_sqrt_R50',NEXT_OUT/'LOSS_WCE_sqrt/comparison/val_metrics.json')]:
    if path.exists():comparison.append(dict(method=name,**json.loads(path.read_text())))
summary=pd.DataFrame(comparison)
summary['delta_macro_f1_pp']=100*(summary.macro_f1-base_scores['macro_f1'])
summary.to_csv(NEXT_OUT/'comparison_validation.csv',index=False)
display(summary)
# Lưu mã triển khai; cache và checkpoint vẫn ở Output nhưng không nhét vào ZIP báo cáo.
for name in ['extract_features','fit_svm_grid','run_weighted_ce','preprocess_resize_only','next_report']:
    if name in globals():(NEXT_OUT/f'{name}.py').write_text(inspect.getsource(globals()[name]),encoding='utf-8')
archive=NEXT_OUT.parent/'ISIC2019_R50_SVM_Loss_v1_reports.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(NEXT_OUT.rglob('*')):
        if p.is_file() and p.suffix not in ['.pt','.npy','.joblib']:
            z.write(p,str(p.relative_to(NEXT_OUT)))
print('Gửi ZIP này + notebook đã chạy:',archive)
print('Giữ toàn bộ Output để dùng lại encoder/cache/model. SVM cần cả checkpoint B00 và pipeline scaler+SVM.')


,method,accuracy,macro_f1,balanced_accuracy,delta_macro_f1_pp
0,B00_R50_CE,0.737692,0.552115,0.540352,0.000000
1,B00_features_LinearSVM,0.734147,0.542479,0.507748,-0.963521
2,L01_WCE_sqrt_R50,0.716424,0.527186,0.541795,-2.492906


Gửi ZIP này + notebook đã chạy: /kaggle/working/ISIC2019_R50_SVM_Loss_v1_reports.zip
Giữ toàn bộ Output để dùng lại encoder/cache/model. SVM cần cả checkpoint B00 và pipeline scaler+SVM.
